# 04. Власна ознака: Custom Temporal Operator

**Етап 04 з 13** · NASA C-MAPSS FD001 · RUL prediction


## 1. Дані

In [ ]:
import mlflow

NGROK_URL = "https://stage-broom-alkalize.ngrok-free.dev"
os.environ["MLFLOW_TRACKING_SERVER_CERT_VERIFY"] = "false"
os.environ["HTTP_HEADER_NGROK_SKIP_BROWSER_WARNING"] = "true"

mlflow.set_tracking_uri(NGROK_URL)
mlflow.set_experiment('predictive_maintenance_colab')

In [3]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBRegressor

DATA_DIR = Path(os.environ.get("FD001_DIR", "/content/drive/MyDrive/predictive_maintenance"))

SENSORS = [f"sensor_{i}" for i in range(1, 22)]
COLUMNS = ["unit_id", "cycle", "setting_1", "setting_2", "setting_3"] + SENSORS

train = pd.read_csv(DATA_DIR / "train_FD001.txt", sep=r"\s+", header=None, names=COLUMNS)

# RUL = останній цикл двигуна - поточний цикл
train["RUL"] = train.groupby("unit_id")["cycle"].transform("max") - train["cycle"]

## 2. Ознаки

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
def custom_temporal_operator(
    values,
    current_idx,
    n_points=4,
    base_step=12,
    min_step=2,
    max_step=15,
    shrink=0.75,
    local_window=5,
    gamma=2.0,
):
    """
    CTO v1 для однієї точки: адаптивно йде назад у минуле й повертає
    (trend, rate_std, curvature, mean_step). Використовує лише значення до current_idx.
    """

    if current_idx < local_window + min_step:
        return np.nan, np.nan, np.nan, np.nan

    idx = current_idx
    rates = []
    steps = []
    step = base_step

    for _ in range(n_points - 1):
        start = max(0, idx - local_window)
        local_history = values[start:idx + 1]

        if len(local_history) < 2:
            break

        # Локальна активність, нормована на локальний масштаб сигналу
        local_activity = np.mean(np.abs(np.diff(local_history)))
        local_scale = np.std(local_history) + 1e-8
        activity_score = local_activity / local_scale

        # Велика активність -> менший крок назад
        adaptive_step = step / (1 + gamma * activity_score)
        adaptive_step = int(round(adaptive_step))
        adaptive_step = np.clip(adaptive_step, min_step, max_step)

        previous_idx = idx - adaptive_step
        if previous_idx < 0:
            break

        # Швидкість зміни на реальному часовому проміжку
        delta = values[idx] - values[previous_idx]
        rates.append(delta / adaptive_step)
        steps.append(adaptive_step)

        idx = previous_idx
        step = max(min_step, step * shrink)

    if len(rates) < 2:
        return np.nan, np.nan, np.nan, np.nan

    rates = np.array(rates)
    steps = np.array(steps)

    return (
        np.mean(rates),                # загальний напрям зміни
        np.std(rates),                 # різноманітність швидкостей
        np.mean(np.diff(rates)),       # зміна самої швидкості (curvature)
        np.mean(steps),                # середня довжина кроку
    )

In [6]:
CTO_V1_SUFFIXES = ["cto_trend", "cto_rate_std", "cto_curvature", "cto_mean_step"]


def add_custom_temporal_features(df, sensor="sensor_11"):
    """
    Додає 4 ознаки CTO v1 для одного сенсора.
    Кожен двигун обробляється окремо, за зростанням cycle.
    """

    df = df.copy()
    cols = [f"{sensor}_{suffix}" for suffix in CTO_V1_SUFFIXES]
    feats = pd.DataFrame(np.nan, index=df.index, columns=cols)

    for _, group in df.groupby("unit_id"):
        group = group.sort_values("cycle")
        values = group[sensor].to_numpy()

        rows = [custom_temporal_operator(values, i) for i in range(len(group))]
        feats.loc[group.index, cols] = np.array(rows, dtype=float)

    return pd.concat([df, feats], axis=1)

In [7]:
CTO_V2_SUFFIXES = [
    "trend",
    "expected_trend",
    "innovation",
    "curvature",
    "multiscale_disagreement",
    "cumulative_deviation",
    "path_span",
]


def add_cto_v2_features(
    df,
    sensors=("sensor_11",),
    short_window=5,
    long_window=15,
    depth=4,
    base_step=12,
    min_step=2,
    max_step=18,
    shrink=0.78,
    gamma=1.5,
    history_decay=0.75,
    eps=1e-8,
):
    """
    CTO v2 для заданих сенсорів: для кожного cycle рахує 7 причинних ознак
    по адаптивному шляху назад A <- B <- C <- ...
    Двигуни обробляються окремо; використовується лише минуле кожної точки.
    """

    df = df.copy()

    def linear_slope(values):
        """Нахил прямої МНК за індексом часу."""
        n = len(values)
        if n < 2:
            return 0.0

        t = np.arange(n, dtype=float)
        y = np.asarray(values, dtype=float)
        t_centered = t - t.mean()
        denominator = np.sum(t_centered ** 2)
        if denominator < eps:
            return 0.0

        return np.sum(t_centered * (y - y.mean())) / denominator

    def causal_state(values, idx):
        """Стан точки idx за її минулою історією (short/long тренди, activity)."""
        short_hist = values[max(0, idx - short_window + 1):idx + 1]
        long_hist = values[max(0, idx - long_window + 1):idx + 1]

        long_slope = linear_slope(long_hist)
        # Розбіжність короткого і довгого масштабів
        disagreement = linear_slope(short_hist) - long_slope

        volatility = np.std(np.diff(long_hist)) if len(long_hist) >= 3 else 0.0

        local_scale = np.std(long_hist)
        if local_scale < eps:
            activity = 0.0
        else:
            activity = (abs(disagreement) + volatility) / (local_scale + eps)
        activity = min(activity, 10.0)

        return {"long_slope": long_slope, "disagreement": disagreement, "activity": activity}

    for sensor in sensors:
        for suffix in CTO_V2_SUFFIXES:
            df[f"{sensor}_cto2_{suffix}"] = np.nan

        for _, group in df.groupby("unit_id", sort=False):
            group = group.sort_values("cycle")
            values = group[sensor].to_numpy(dtype=float)
            row_indices = group.index.to_numpy()

            for current_idx in range(len(values)):
                if current_idx < max(short_window, min_step):
                    continue

                idx = current_idx
                current_base_step = float(base_step)

                rates, expected_rates, innovations = [], [], []
                disagreements, steps, weights = [], [], []

                for level in range(depth):
                    state_current = causal_state(values, idx)

                    # Велика активність -> менший крок
                    adaptive_step = current_base_step / (1.0 + gamma * state_current["activity"])
                    adaptive_step = int(np.round(adaptive_step))
                    adaptive_step = int(np.clip(adaptive_step, min_step, max_step))

                    previous_idx = idx - adaptive_step
                    if previous_idx < 0:
                        break

                    # Історія B дає очікуваний тренд після B;
                    # innovation = реальний рух B -> A мінус очікування
                    state_previous = causal_state(values, previous_idx)
                    actual_rate = (values[idx] - values[previous_idx]) / adaptive_step
                    expected_rate = state_previous["long_slope"]

                    rates.append(actual_rate)
                    expected_rates.append(expected_rate)
                    innovations.append(actual_rate - expected_rate)
                    disagreements.append(state_previous["disagreement"])
                    steps.append(adaptive_step)
                    weights.append(history_decay ** level)

                    idx = previous_idx
                    current_base_step = max(min_step, current_base_step * shrink)

                if len(rates) < 2:
                    continue

                rates = np.asarray(rates, dtype=float)
                expected_rates = np.asarray(expected_rates, dtype=float)
                innovations = np.asarray(innovations, dtype=float)
                disagreements = np.asarray(disagreements, dtype=float)
                steps = np.asarray(steps, dtype=float)
                weights = np.asarray(weights, dtype=float)
                weights = weights / (weights.sum() + eps)

                trend = np.sum(weights * rates)
                expected_trend = np.sum(weights * expected_rates)
                innovation = np.sum(weights * innovations)

                # Різниця сусідніх переходів, ваги без найстарішого
                rate_changes = rates[:-1] - rates[1:]
                curvature_weights = weights[:-1]
                curvature_weights = curvature_weights / (curvature_weights.sum() + eps)
                curvature = np.sum(curvature_weights * rate_changes)

                multiscale_disagreement = np.sum(weights * np.abs(disagreements))
                cumulative_deviation = np.sum(innovations * steps * weights)
                path_span = np.sum(steps)

                row_idx = row_indices[current_idx]
                values_by_suffix = {
                    "trend": trend,
                    "expected_trend": expected_trend,
                    "innovation": innovation,
                    "curvature": curvature,
                    "multiscale_disagreement": multiscale_disagreement,
                    "cumulative_deviation": cumulative_deviation,
                    "path_span": path_span,
                }
                for suffix in CTO_V2_SUFFIXES:
                    df.loc[row_idx, f"{sensor}_cto2_{suffix}"] = values_by_suffix[suffix]

    return df

In [8]:
TOP8_SENSORS = [
    "sensor_11", "sensor_4", "sensor_12", "sensor_7",
    "sensor_9", "sensor_15", "sensor_20", "sensor_21",
]

# Один датафрейм з усіма CTO-ознаками (причинні обчислення всередині двигуна)
feat = add_custom_temporal_features(train, sensor="sensor_11")
feat = add_cto_v2_features(feat, sensors=TOP8_SENSORS)

BASE = ["cycle", "setting_1", "setting_2", "setting_3"] + SENSORS
CTO_V1 = [f"sensor_11_{suffix}" for suffix in CTO_V1_SUFFIXES]
CTO_V2_SENSOR11 = [f"sensor_11_cto2_{suffix}" for suffix in CTO_V2_SUFFIXES]
CTO_V2_TOP8 = [f"{s}_cto2_{suffix}" for s in TOP8_SENSORS for suffix in CTO_V2_SUFFIXES]


def cto_group(suffix):
    """Одна CTO v2-ознака для всіх top-8 сенсорів."""
    return [f"{s}_cto2_{suffix}" for s in TOP8_SENSORS]


EXPECTED = cto_group("expected_trend")
DISAGREEMENT = cto_group("multiscale_disagreement")
CUMULATIVE = cto_group("cumulative_deviation")
PATH = cto_group("path_span")

## 3. Експерименти

Усі конфігурації навчаються на одному розбитті з однаковими параметрами XGBoost.

In [9]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(feat, groups=feat["unit_id"]))

train_df = feat.iloc[train_idx].copy()
val_df = feat.iloc[val_idx].copy()

print("Train engines:", train_df["unit_id"].nunique(), "| rows:", len(train_df))
print("Validation engines:", val_df["unit_id"].nunique(), "| rows:", len(val_df))

Train engines: 80 | rows: 16561
Validation engines: 20 | rows: 4070


In [10]:
XGB_PARAMS = {
    "n_estimators": 300,
    "learning_rate": 0.012074507989018332,
    "max_depth": 4,
    "min_child_weight": 3.2298041995296414,
    "subsample": 0.7900204897773602,
    "colsample_bytree": 0.9223090698240903,
    "reg_alpha": 0.7459635164536705,
    "reg_lambda": 4.416722489003329,
    "gamma": 0.2695542352154229,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
}

# Конфігурації наборів ознак: порядок колонок зберігається як в оригінальних експериментах
EXPERIMENTS = {
    "baseline": BASE,
    "cto_v1_sensor_11": BASE + CTO_V1,
    "cto_v2_sensor_11": BASE + CTO_V2_SENSOR11,
    "expected_only": BASE + EXPECTED,
    "expected_disagreement": BASE + EXPECTED + DISAGREEMENT,
    "expected_disagreement_cumulative": BASE + EXPECTED + DISAGREEMENT + CUMULATIVE,
    "expected_disagreement_cumulative_path": BASE + EXPECTED + DISAGREEMENT + CUMULATIVE + PATH,
    "full_cto": BASE + CTO_V2_TOP8,
}

In [13]:
def run_experiments(experiments, train_df, val_df):
    """Навчає XGBoost на кожному наборі ознак і повертає MAE, RMSE, R2 на validation."""
    rows = []
    for name, features in experiments.items():
        model = XGBRegressor(**XGB_PARAMS)
        model.fit(train_df[features], train_df["RUL"])

        pred = np.clip(model.predict(val_df[features]), 0, None)
        y_val = val_df["RUL"]

        metrics = {
            "MAE": mean_absolute_error(y_val, pred),
            "RMSE": np.sqrt(mean_squared_error(y_val, pred)),
            "R2": r2_score(y_val, pred),
        }

        rows.append({
            "experiment": name,
            "n_features": len(features),
            **metrics
        })

        run_name = f"Cutom Features | {name}"
        with mlflow.start_run(run_name=run_name):
          mlflow.log_params({
              "experiment_name": name,
              "model_type": model.__class__.__name__,
              "num_features": len(features),
              "features_list": ", ".join(features),
              "prediction_type": "Clipped at 0",
              **XGB_PARAMS
          })

          mlflow.log_metrics(metrics)

    return pd.DataFrame(rows)

## 4. Результати

In [14]:
results = run_experiments(EXPERIMENTS, train_df, val_df)

baseline = results.loc[results["experiment"] == "baseline"].iloc[0]
results["ΔMAE"] = results["MAE"] - baseline["MAE"]
results["ΔRMSE"] = results["RMSE"] - baseline["RMSE"]
results["ΔR2"] = results["R2"] - baseline["R2"]

# TODO(review): найкраща конфігурація обирається за MAE на validation,
# тому оцінка для неї трохи оптимістична. Потрібен окремий тест або CV.
results = results.sort_values("MAE").reset_index(drop=True)
display(results)

🏃 View run Cutom Features | baseline at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1/runs/4d293c43d3f042cca2c29179b8e2fb2e
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1
🏃 View run Cutom Features | cto_v1_sensor_11 at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1/runs/c4f52dd9840a4305b647dcacf3fff856
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1
🏃 View run Cutom Features | cto_v2_sensor_11 at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1/runs/301c640cfa0343fa9d2e05f6f15c46d7
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1
🏃 View run Cutom Features | expected_only at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1/runs/e3f0fc5abed64de18c4fee429f9becde
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1
🏃 View run Cutom Features | expected_disagreement at: https://stage-broom-alkalize.ngrok-free.d

,experiment,n_features,MAE,RMSE,R2,ΔMAE,ΔRMSE,ΔR2
0,expected_disagreement,41,22.304037,29.113475,0.803350,-0.767637,-0.745156,0.010195
1,expected_disagreement_cumulative,49,22.313391,29.099292,0.803542,-0.758284,-0.759339,0.010387
2,expected_disagreement_cumulative_path,57,22.318649,29.105723,0.803455,-0.753025,-0.752908,0.010300
3,full_cto,81,22.347841,29.144630,0.802929,-0.723833,-0.714000,0.009774
4,expected_only,33,22.349064,29.182493,0.802417,-0.722610,-0.676138,0.009262
5,cto_v2_sensor_11,32,22.650375,29.395221,0.799526,-0.421299,-0.463410,0.006371
6,baseline,25,23.071674,29.858631,0.793155,0.000000,0.000000,0.000000
7,cto_v1_sensor_11,29,23.078659,29.838785,0.793430,0.006985,-0.019846,0.000275


## Висновки

Значення з оригінальних виводів ноутбука (validation, 80/20 за двигунами):

- **Baseline** (25 базових ознак): MAE 23.0717, RMSE 29.8586, R² 0.7932.
- **CTO v1** (sensor_11, 4 ознаки): MAE 23.0787 (+0.0070), RMSE 29.8388, R² 0.7934. Суттєвого покращення немає.
- **CTO v2, sensor_11** (7 ознак): MAE 22.6504 (−0.4213), RMSE 29.3952, R² 0.7995.
- **CTO v2, top-8, повний набір** (56 ознак, разом 81): MAE 22.3478 (−0.7238), RMSE 29.1446, R² 0.8029.
- **Найкращий варіант — expected_disagreement** (8 expected_trend + 8 multiscale_disagreement, разом 41): MAE 22.3040 (−0.7676), RMSE 29.1135 (−0.7452), R² 0.803350 (+0.010195).

Додавання cumulative_deviation (MAE 22.3134) і path_span (MAE 22.3186) не покращує результат порівняно з expected_disagreement (22.3040).